# Scripture Everywhere AI
## One Brain. Every Digital Moment.

This public notebook documents and demonstrates the shared context engine submitted to **Scripture in New Frontiers**. It is fully runnable without private API keys; live mode uses Gloo AI Studio and YouVersion Platform API through the repository backend.


## The problem
People experience high-friction moments inside workouts, games, IDEs, communities, and creator tools. Existing Bible experiences usually require them to leave the moment. Our system makes Scripture context-aware, safe, private, and native to the surface where the moment occurs.


In [ ]:
from dataclasses import dataclass, asdict
from typing import Any
import pandas as pd

@dataclass
class ContextEvent:
    source: str
    moment_type: str
    metrics: dict[str, Any]
    privacy: str = 'private'
    opted_in: bool = True
    locale: str = 'en'


## Five frontiers, one event contract
Every connector emits the same normalized structure. The intelligence and safety layers stay shared while only the input and delivery surface change.


In [ ]:
events = [
    ContextEvent('wearable', 'effort_peak', {'heart_rate': 170, 'effort': 0.85, 'minutes': 18}),
    ContextEvent('gaming', 'repeat_failure', {'failures': 8, 'session_minutes': 41}),
    ContextEvent('ide', 'build_loop', {'failed_builds': 27, 'focus_minutes': 194}),
    ContextEvent('social', 'distress_signal', {'confidence': 0.91}, privacy='public'),
    ContextEvent('creator', 'toxicity_spike', {'pressure_index': 94, 'live_minutes': 102}),
]
pd.DataFrame([asdict(e) for e in events])


## Deterministic judge-safe demonstration
The hackathon demo remains reproducible without credentials. In live mode, Gloo returns the need/theme/tone/safety decision and YouVersion retrieves the passage text. The fallback below mirrors the same typed contract without pretending to be a live API call.


In [ ]:
THEMES = {
    'wearable': ('endurance', 'strength', 'concise', 'ISA.40.31'),
    'gaming': ('encouragement', 'perseverance', 'teammate', 'JAS.1.12'),
    'ide': ('clarity', 'wisdom', 'calm', 'JAS.1.5'),
    'social': ('support', 'comfort', 'gentle', 'PSA.34.18'),
    'creator': ('grounding', 'restraint', 'steady', 'PRO.15.1'),
}

def decide(event: ContextEvent) -> dict[str, Any]:
    if not event.opted_in:
        return {'suppressed': True, 'reason': 'consent_required'}
    need, theme, tone, passage_id = THEMES[event.source]
    public_allowed = not (event.source == 'social' and event.privacy == 'public')
    return {
        'suppressed': False,
        'need': need,
        'theme': theme,
        'tone': tone,
        'safe_to_deliver': True,
        'public_delivery_allowed': public_allowed,
        'passage_id': passage_id,
    }

decisions = [decide(e) for e in events]
pd.DataFrame(decisions)


## Wearable hero story
The primary demo is intentionally one human story rather than five disconnected mini-apps: a runner reaches a difficult physiological moment, Gloo identifies a concise need for strength, YouVersion supplies Scripture, and the delivery policy waits for a safe recovery window before a quiet haptic cue.


In [ ]:
hero = events[0]
decision = decide(hero)
pipeline = pd.DataFrame([
    {'stage': 'Context', 'proof': hero.metrics},
    {'stage': 'Gloo AI Studio', 'proof': {k: decision[k] for k in ('need','theme','tone','safe_to_deliver')}},
    {'stage': 'YouVersion', 'proof': {'passage_id': decision['passage_id'], 'source': 'Platform API in live mode'}},
    {'stage': 'Delivery policy', 'proof': {'surface': 'wearable_card', 'timing': 'recovery_window', 'privacy': 'private'}},
])
pipeline


## Safety and trust gates
- Explicit user opt-in is required.
- Sensitive public social events never auto-post Scripture.
- Crisis/self-harm signals suppress normal delivery and route to human support.
- The model does not diagnose or claim divine certainty.
- Cooldowns and dismiss controls prevent notification fatigue.
- Raw biometrics and private text are not retained by the prototype.


In [ ]:
checks = {
    'consent_gate': decide(ContextEvent('wearable','effort_peak',{}, opted_in=False))['suppressed'],
    'public_social_autopost_blocked': not decide(events[3])['public_delivery_allowed'],
    'allowed_theme_only': all(d.get('theme') in {'strength','perseverance','wisdom','comfort','restraint'} for d in decisions),
}
assert all(checks.values())
checks


## Technical architecture
```text
Connector Event → Context Normalizer → Gloo AI Studio → Theme Allowlist → YouVersion Passage API → Delivery Policy → Native Surface
```

Repository: https://github.com/o-yutaka/popopo  
Public demo target: https://o-yutaka.github.io/popopo/
